In [ ]:
# Find the pipeline package without walking the DICOM tree.
import sys, os, time, gc, json
from pathlib import Path

T0 = time.time()
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

def walk(root="/kaggle/input", max_depth=8, suffixes=None):
    base = len(Path(root).parts)
    for dirpath, dirnames, filenames in os.walk(root):
        here = Path(dirpath)
        if len(here.parts) - base >= max_depth:
            dirnames[:] = []
        else:
            dirnames[:] = [d for d in dirnames if not d.lower().endswith(("_series", "_images"))]
        for f in filenames:
            if suffixes is None or Path(f).suffix.lower() in suffixes:
                yield here / f

PKG = next((p.parent.parent for p in walk(suffixes=(".py",))
            if p.name == "targets.py" and p.parent.name == "rsna_knee"), None)
if PKG is None:
    raise FileNotFoundError("attach the rsna-knee-pipeline dataset (v2, it must contain rsna_knee/targets.py)")
sys.path.insert(0, str(PKG))

import numpy as np
import pandas as pd
from rsna_knee.report_labels import TARGETS
log(f"pipeline: {PKG}")


In [ ]:
# GPU notebook. Inputs: rsna-knee-pipeline, the three cache v3 shards and the five fold notebooks
# knee r50v3 fold0 ... fold4. Writes slot_attention.csv: for every fold, the learned slot prior and the
# mean finding-to-slot attention on that fold's held-out studies.
import torch
from torch.utils.data import DataLoader
from rsna_knee.cache3 import load_cache3, copy_local
from rsna_knee.train2 import TrainConfigV2, build_model, make_dataset
from rsna_knee.infer2 import find_checkpoints
from rsna_knee.targets import file_digest
TAG = "r50v3"

FOLDS_CSV = next(p for p in walk(suffixes=(".csv",)) if p.name == "train_folds.csv")
train = pd.read_csv(FOLDS_CSV)
CACHE_DIRS = copy_local(sorted({p.parent for p in walk(suffixes=(".npy",)) if p.name.startswith("c3_train_")}), log=log)
array, mask, _ = load_cache3(CACHE_DIRS, "train", train.StudyInstanceUID.tolist())
ckpts = {p.name: p for p in find_checkpoints(walk).get(TAG, [])}
pd.DataFrame([{"checkpoint": n, "sha256": file_digest(p), "bytes": p.stat().st_size} for n, p in ckpts.items()]
             ).to_csv("/kaggle/working/checkpoint_manifest.csv", index=False)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rows = []
for k in range(5):
    p = ckpts.get(f"v2_{TAG}_fold{k}.pt")
    if p is None:
        log(f"fold {k}: checkpoint not mounted"); continue
    st = torch.load(p, map_location="cpu", weights_only=False)
    cfg = TrainConfigV2(**{a: b for a, b in st["config"].items() if a in TrainConfigV2.__dataclass_fields__})
    cfg.encoder_weights = None
    m = build_model(cfg, pretrained=False, log=lambda *_: None)
    m.load_state_dict({a: b.float() if b.is_floating_point() else b for a, b in st["model"].items()})
    m.to(dev).eval()
    caught = []
    head = m.head
    def hook(mod, inp, out):
        slots, present = inp
        h = mod.proj(slots) + mod.slot_id.unsqueeze(0)
        lg = torch.einsum("bsd,od->bos", h.float(), mod.query.float()) / mod.hidden ** 0.5
        if mod.prior is not None:
            lg = lg + mod.prior.float().unsqueeze(0)
        lg = lg.masked_fill(present.unsqueeze(1) < 0.5, float("-inf"))
        caught.append(torch.nan_to_num(torch.softmax(lg, -1), nan=0.0).cpu().numpy())
    hk = head.register_forward_hook(hook)
    va = np.flatnonzero(train.fold.values == k)
    dl = DataLoader(make_dataset(array, mask, va, None, None, cfg, train=False), batch_size=2, num_workers=2)
    with torch.no_grad():
        for b in dl:
            with torch.autocast("cuda", enabled=dev.type == "cuda"):
                m(b["x"].to(dev), b["present"].to(dev))
    hk.remove()
    att = np.concatenate(caught).mean(0)
    prior = head.prior.detach().float().cpu().numpy() if head.prior is not None else np.zeros_like(att)
    for j, t in enumerate(TARGETS):
        for s in range(6):
            rows.append({"fold": k, "finding": t, "slot": s, "attention": float(att[j, s]),
                         "prior": float(prior[j, s])})
    log(f"fold {k}: {len(va)} studies")
pd.DataFrame(rows).to_csv("/kaggle/working/slot_attention.csv", index=False)
log("done")
